# 自作モジュール：公開 API とテスト

対応する本文: [modules.tex](../chapters/modules.tex)。各問を「考え方 → 実装 → 確認」の順で解説する。上から全セルを実行すると、入力の作成から結果の検証までを再現できる。生成物は専用の出力先へ保存する。

In [1]:
import sys
from pathlib import Path

# リポジトリ直下と notebooks/ のどちらから起動しても同じ場所を使う。
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "main.tex").exists() and (p / "chapters").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("教材のフォルダー内で Notebook を起動してください。")
WORK = ROOT / "notebooks" / "_outputs" / "11_modules"
WORK.mkdir(parents=True, exist_ok=True)
print("Python", sys.version.split()[0])
print("出力先:", WORK.relative_to(ROOT))

Python 3.12.14
出力先: notebooks/_outputs/11_modules


## 問1：src レイアウトのパッケージを作る

本文では `uv init --lib signal_tools` を使う。`--lib` は import 用のライブラリ構成を指定し、`src` はソースをプロジェクト直下から分離する。`name` は配布名、`requires-python` は対応する Python の範囲である。

下では通信なしで全セルを実行できるよう、同等の最小構成を作る。通常インストールと editable install を比較するため、ビルドバックエンドに setuptools を使用する。本文の uv が生成する uv_build と役割は同じだが、設定はバックエンドごとに異なる。

In [2]:
import subprocess
import textwrap

PROJECT = WORK / "signal_tools"
PACKAGE = PROJECT / "src" / "signal_tools"
PACKAGE.mkdir(parents=True, exist_ok=True)
(PROJECT / "tests").mkdir(exist_ok=True)
(PROJECT / "pyproject.toml").write_text(textwrap.dedent('''[build-system]
requires = ["setuptools>=64", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "signal-tools"
version = "0.1.0"
requires-python = ">=3.10"
dependencies = ["numpy>=2"]

[dependency-groups]
dev = ["pytest", "ruff"]

[tool.setuptools.packages.find]
where = ["src"]
'''), encoding="utf-8")
(PACKAGE / "__init__.py").write_text("from .statistics import mean_signal\n\n__all__ = ['mean_signal']\n", encoding="utf-8")
(PACKAGE / "py.typed").touch()
print((PROJECT / "pyproject.toml").read_text())

[build-system]
requires = ["setuptools>=64", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "signal-tools"
version = "0.1.0"
requires-python = ">=3.10"
dependencies = ["numpy>=2"]

[dependency-groups]
dev = ["pytest", "ruff"]

[tool.setuptools.packages.find]
where = ["src"]



## 問2・問3：関数、型ヒント、docstring、テストをまとめる

公開関数は通常入力と空入力を検証する。型ヒントを書くだけでは実行時の型検査は行われない。docstring は利用者が必要な入力、返り値、例外条件を説明する。

In [3]:
source = textwrap.dedent('''from collections.abc import Sequence


def mean_signal(values: Sequence[float]) -> float:
    """信号列の算術平均を返す。

    Args:
        values: 数値の列。
    Returns:
        入力値の算術平均。
    Raises:
        ValueError: 入力列が空の場合。
    """
    if len(values) == 0:
        raise ValueError("入力列が空です。")
    return sum(values) / len(values)
''')
(PACKAGE / "statistics.py").write_text(source, encoding="utf-8")
(PROJECT / "tests" / "test_statistics.py").write_text(textwrap.dedent('''import pytest
from signal_tools import mean_signal


def test_mean_signal():
    assert mean_signal([1.0, 2.0, 3.0]) == pytest.approx(2.0)


def test_empty_input():
    with pytest.raises(ValueError):
        mean_signal([])
'''), encoding="utf-8")

225

## 問4：2つの環境へローカルにインストールする

比較用の venv は教材の環境に導入済みの pytest やビルドツールを参照する。自作パッケージ自体は各 venv に別々にインストールする。`--no-index --no-deps --no-build-isolation` を用いてネットワークアクセスを行わない。この指定はビルドと依存関係を事前に用意した実習のためのものである。

通常インストールはソースをコピーして配置し、editable install は作業中のソースを参照する。すでに import したモジュールのキャッシュと混同しないよう、比較のたびに新しい Python プロセスを起動する。

In [4]:
# Notebook 用 venv の site-packages にあるビルド・検証ツールを共有する。
import site
import venv

shared_site = next(p for p in site.getsitepackages() if Path(p).is_dir() and "site-packages" in p)
environments = {}
for name, editable in [("regular", False), ("editable", True)]:
    env_dir = WORK / name
    venv.EnvBuilder(with_pip=True).create(env_dir)
    executable = env_dir / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")
    site_dir = subprocess.run([str(executable), "-c", "import site; print(site.getsitepackages()[0])"],
                              capture_output=True, text=True, check=True).stdout.strip()
    (Path(site_dir) / "textbook_dependencies.pth").write_text(shared_site + "\n", encoding="utf-8")
    command = [str(executable), "-m", "pip", "install", "--no-index", "--no-deps", "--no-build-isolation", "--quiet"]
    if editable:
        command.append("-e")
    subprocess.run(command + [str(PROJECT)], cwd=WORK, capture_output=True, text=True, check=True)
    environments[name] = executable
for name, executable in environments.items():
    result = subprocess.run([str(executable), "-c", "from signal_tools import mean_signal; print(mean_signal([1,2,3]))"],
                            cwd=WORK, capture_output=True, text=True, check=True)
    assert float(result.stdout) == 2.0
    print(name, result.stdout.strip())
tested = subprocess.run([str(environments["editable"]), "-m", "pytest", "-q", str(PROJECT / "tests")],
                        cwd=WORK, capture_output=True, text=True, check=True)
print(tested.stdout)
assert "2 passed" in tested.stdout
help_result = subprocess.run([str(environments["editable"]), "-c", "from signal_tools import mean_signal; help(mean_signal)"],
                             cwd=WORK, capture_output=True, text=True, check=True)
print(help_result.stdout)

regular 2.0
editable 2.0
..                                                                       [100%]
2 passed in 0.00s



Help on function mean_signal in module signal_tools.statistics:

mean_signal(values: collections.abc.Sequence[float]) -> float
    信号列の算術平均を返す。

    Args:
        values: 数値の列。
    Returns:
        入力値の算術平均。
    Raises:
        ValueError: 入力列が空の場合。




In [5]:
# 関数の動作を変え、再起動したプロセスで差を確認した後、元へ戻す。
try:
    changed = source.replace("return sum(values) / len(values)", "return -99.0")
    (PACKAGE / "statistics.py").write_text(changed, encoding="utf-8")
    observed = {}
    for name, executable in environments.items():
        result = subprocess.run([str(executable), "-c", "from signal_tools import mean_signal; print(mean_signal([1,2,3]))"],
                                cwd=WORK, capture_output=True, text=True, check=True)
        observed[name] = float(result.stdout)
    assert observed == {"regular": 2.0, "editable": -99.0}
    print(observed)
finally:
    (PACKAGE / "statistics.py").write_text(source, encoding="utf-8")

{'regular': 2.0, 'editable': -99.0}


## 問5：実行時と開発時の依存関係を分ける

uv を使う場合は `uv add numpy` と `uv add --dev pytest ruff` を実行する。上の pyproject.toml でも同じ分類を確認できる。`.venv`、キャッシュ、配布生成物は再生成できるため Git から除く。`uv.lock` は共同開発の環境をそろえるためコミットする方針が考えられる。一方、利用者への配布依存関係は pyproject.toml で定義する。

通常の開発ではパッケージをインストールした環境で `uv run pytest` と `uv run ruff check .` を実行する。

In [6]:
(PROJECT / ".gitignore").write_text(".venv/\n__pycache__/\n.pytest_cache/\ndist/\n", encoding="utf-8")
result = subprocess.run([sys.executable, "-m", "ruff", "check", str(PROJECT / "src"), str(PROJECT / "tests")],
                        capture_output=True, text=True, check=True)
print(result.stdout)
print((PROJECT / ".gitignore").read_text())

All checks passed!

.venv/
__pycache__/
.pytest_cache/
dist/

